# 03 — Score each 10-K for tariff and import exposure

**Goal:** turn each 10-K into numbers that measure how much the firm talks about tariffs and about relying on imports or foreign suppliers.

**Method (v0): dictionary / keyword scores.** For each pattern we count matches, then scale by document length:

$$\text{score} = \frac{\text{matches}}{\text{total words}} \times 10{,}000 \quad (\text{matches per 10,000 words})$$

**Why keywords and not an LLM (for v0):** keyword counts are fully transparent and can't "know" what happened after April 2025. An LLM might have learned about the tariffs in training, which is the **LLM leakage** problem. That makes keywords the clean baseline. We may add an LLM score later as a comparison.

**Design choices an auditor will look for:**
- We match **phrases** with regular expressions, not single words. So "trade war" and "foreign suppliers" can match.
- We do **not** count the bare word "duty", because "fiduciary duty" would count. We only count "import / customs / anti-dumping / countervailing duty".
- `\bimport(s|ed|ing|ers?)?\b` matches "imports" but **not** "important".
- We drop the hidden XBRL header (`ix:header`) in modern 10-Ks, which is machine data, not text a reader sees.
- We save the count for **every pattern** and the **word count**, so anyone can check whether the score just measures how long a 10-K is.

| | |
|---|---|
| **Inputs** | `data/raw/edgar_tariffs/*.html`, `data/processed/tenk_manifest.csv` |
| **Outputs** | `data/processed/filing_scores.csv`<br>`outputs/keyword_snippets_sample.csv` (random matched sentences for a hand check) |


In [ ]:
# --- Setup: find the repo root and load shared settings from code/config.py ---
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "run_all.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Can't find the repo root. Run this from inside the project folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code"))

import config as cfg
print("Repo root:", ROOT)

In [ ]:
import re
import random
import pandas as pd
from bs4 import BeautifulSoup

TARIFF_PATTERNS = {
    "tariff":         r"\btariffs?\b",
    "trade_policy":   r"\btrade (?:policy|policies|wars?|restrictions?|tensions?|disputes?|barriers?)\b",
    "duties":         r"\b(?:import|customs|anti-?dumping|countervailing) dut(?:y|ies)\b",
    "section_301_232": r"\bsection (?:301|232)\b",
}
IMPORT_PATTERNS = {
    "import_words":      r"\bimport(?:s|ed|ing|ers?)?\b",
    "foreign_suppliers": r"\b(?:foreign|overseas|international|offshore) (?:suppliers?|vendors?|manufactur\w*|sourcing|production|factories)\b",
    "contract_mfg":      r"\bcontract manufactur\w*\b",
}
CHINA_PATTERNS = {
    "china": r"\b(?:china|chinese|prc)\b",
}

## Quick unit test of the patterns (hand-checkable)

In [ ]:
def count(pattern, text):
    return len(re.findall(pattern, text))

test = ("tariffs and a new tariff. trade war risk. fiduciary duty. import duties apply. "
        "imports from foreign suppliers. it is important. contract manufacturers in china.")
assert count(TARIFF_PATTERNS["tariff"], test) == 2
assert count(TARIFF_PATTERNS["trade_policy"], test) == 1
assert count(TARIFF_PATTERNS["duties"], test) == 1          # "fiduciary duty" NOT counted
assert count(IMPORT_PATTERNS["import_words"], test) == 2    # "import" + "imports", NOT "important"
assert count(IMPORT_PATTERNS["foreign_suppliers"], test) == 1
assert count(IMPORT_PATTERNS["contract_mfg"], test) == 1
assert count(CHINA_PATTERNS["china"], test) == 1
print("All pattern tests passed.")

## Score every filing

In [ ]:
def html_to_text(path):
    soup = BeautifulSoup(path.read_bytes(), "html.parser")
    for tag in soup.find_all(["script", "style", "ix:header"]):
        tag.decompose()
    text = soup.get_text(" ")
    return re.sub(r"\s+", " ", text).lower()


manifest = pd.read_csv(cfg.MANIFEST_CSV, dtype={"cik": str})
records, snippets = [], []
random.seed(331)

for _, m in manifest.iterrows():
    text = html_to_text(cfg.EDGAR_DIR / m["local_file"])
    n_words = len(re.findall(r"[a-z]+", text))
    row = {"ticker": m["ticker"], "filing_date": m["filing_date"], "word_count": n_words}

    for group, patterns in [("tariff", TARIFF_PATTERNS), ("import", IMPORT_PATTERNS), ("china", CHINA_PATTERNS)]:
        hits = 0
        for name, pat in patterns.items():
            matches = list(re.finditer(pat, text))
            row[f"n_{name}"] = len(matches)
            hits += len(matches)
            # keep up to 2 random examples per pattern for a human to check
            for mt in random.sample(matches, min(2, len(matches))):
                snippets.append({"ticker": m["ticker"], "pattern": name,
                                 "snippet": text[max(0, mt.start() - 120): mt.end() + 120]})
        row[f"{group}_hits"] = hits
        row[f"{group}_score"] = hits / n_words * 10_000 if n_words else None
    records.append(row)
    print(f"{m['ticker']}: {n_words:,} words, tariff_score={row['tariff_score']:.2f}, import_score={row['import_score']:.2f}")

scores = pd.DataFrame(records)
scores.to_csv(cfg.SCORES_CSV, index=False)
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(snippets).to_csv(cfg.OUTPUT_DIR / "keyword_snippets_sample.csv", index=False)
scores

## Checks
A very short "10-K" usually means we saved an index page or exhibit, not the real filing. The length correlation shows whether our score is mostly a document-length effect.

In [ ]:
assert scores["ticker"].is_unique
too_short = scores[scores["word_count"] < 20_000]
if len(too_short):
    print("WARNING: suspiciously short filings (check these by hand):\n", too_short[["ticker", "word_count"]])
print("Correlation of tariff_score with word_count:",
      round(scores["tariff_score"].corr(scores["word_count"]), 3))